# LCCA-v9 exact reproduction, checkpoint archival, and polarity contrast evaluation

This notebook performs one locked run with three sequential goals:

1. Reproduce the completed V9 frozen-DeBERTa experiment without changing any training hyperparameter.
2. Save `head`, `mlp`, and `lcca` adapter/scorer checkpoints for seeds 13, 42, and 71 (nine `.pt` files).
3. If and only if the reproduction audit passes, score all O/A/B arms of the supplied 81-pair polarity contrast set with the unchanged `score_contrast_set.py` metric implementation.

The encoder is frozen. MLP and LCCA each contain exactly 1,050,625 adapter parameters and share each frozen encoder forward pass within a seed. The output ZIP includes the checkpoints, held-out reproduction table, `metrics.json`, the printed markdown table, predictions, configuration, and integrity hashes.

**Input serialization audit.** The completed V9 source code called the tokenizer as `tokenizer(question + option, passage)`. For DeBERTa this yields `[CLS; question; option; SEP; passage; SEP]`, not the passage-first order later written in prose. Exact reproduction requires retaining the executable V9 order for both BoolQ and contrast-set scoring. The notebook records this correction as an artifact so the manuscript can be amended.


In [ ]:
# FIRST CELL TO RUN: Google Drive permission is requested before any long operation.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive connected.')


In [ ]:
# PRE-FLIGHT: upload both files to the Colab session before running all cells.
from pathlib import Path

SCORE_SCRIPT_PATH = Path('/content/score_contrast_set.py')
CONTRAST_SET_PATH = Path('/content/contrast_set.jsonl')
missing_inputs = [str(p) for p in (SCORE_SCRIPT_PATH, CONTRAST_SET_PATH) if not p.is_file()]
if missing_inputs:
    raise FileNotFoundError(
        'Upload these files to /content before training starts: ' + ', '.join(missing_inputs)
    )
print('Contrast inputs found:', SCORE_SCRIPT_PATH.name, CONTRAST_SET_PATH.name)


In [ ]:
# Versions are pinned to avoid Colab's pandas / Hugging Face dependency conflicts.
%pip -q uninstall -y gradio gradio_client
%pip -q install --upgrade --no-cache-dir \
  "pandas==2.2.3" \
  "transformers==4.57.6" \
  "datasets==4.0.0" \
  "huggingface_hub==0.36.2" \
  "accelerate>=1.10,<2" \
  "sentencepiece>=0.2,<0.3" \
  "protobuf>=5.29,<6" \
  "scikit-learn>=1.6,<1.8" \
  "seaborn>=0.13,<0.14" \
  "tabulate>=0.9,<1"


In [ ]:
import gc, json, math, os, random, shutil, time, warnings
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import Dataset, DatasetDict, load_dataset
from huggingface_hub.utils import logging as hf_logging
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from transformers.utils import logging as transformers_logging

warnings.filterwarnings('ignore', category=FutureWarning)
hf_logging.set_verbosity_error()
transformers_logging.set_verbosity_error()
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_DISABLED'] = 'true'

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('GPU bulunamadı. Colab: Runtime > Change runtime type > GPU seçin.')

AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')

try:
    import psutil
    RAM_GB = psutil.virtual_memory().total / (1024 ** 3)
except Exception:
    RAM_GB = None

print('Device:', DEVICE)
print('GPU:', torch.cuda.get_device_name(0))
print('Mixed precision:', AMP_DTYPE)
print('RAM (GB):', round(RAM_GB, 1) if RAM_GB else 'unknown')
if RAM_GB and RAM_GB < 24:
    print('Not: Colab High-RAM atamamış. Bu pilot yine çalışır; veri yükleme biraz daha yavaş olabilir.')


In [ ]:
# ----------------------- LOCKED V9 CONFIGURATION -----------------------
DRIVE_MODEL_DIR = Path('/content/drive/MyDrive/Colab Notebooks/NLP Modeller/deberta-v3-large')
LOCAL_MODEL_DIR = Path('/content/models/deberta-v3-large')
RESULT_DIR = Path('/content/lcca_v9_reproduction_contrast_results')
PROGRESS_DIR = Path('/content/drive/MyDrive/Colab Notebooks/LCCA Results/lcca_v9_reproduction_progress')
CHECKPOINT_DIR = RESULT_DIR / 'checkpoints'
DRIVE_CHECKPOINT_DIR = PROGRESS_DIR / 'checkpoints'
RESUME_FROM_DRIVE = True

DATA_SEED = 2026
SEEDS = [13, 42, 71]
EPOCHS = 3
MAX_LENGTH = 512
LATENT_RANK = 256
LATENT_HEADS = 8
MLP_BOTTLENECK = 2 * LATENT_RANK
BATCH_SIZE = 2
EVAL_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8
TRAIN_LR = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
SHUFFLE_TRIALS = 3
VAL_TARGET = 1075
REPRO_TOLERANCE_POINTS = 0.50

RESULT_DIR.mkdir(parents=True, exist_ok=True)
PROGRESS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(DATA_SEED)

required = ['config.json', 'spm.model', 'tokenizer_config.json']
missing = [name for name in required if not (DRIVE_MODEL_DIR / name).is_file()]
weight_ok = any((DRIVE_MODEL_DIR / name).is_file() for name in ['model.safetensors', 'pytorch_model.bin'])
if missing or not weight_ok:
    raise FileNotFoundError(
        f'Model folder incomplete: {DRIVE_MODEL_DIR}. Missing: {missing}; weight file present: {weight_ok}'
    )

LOCAL_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR, dirs_exist_ok=True)
MODEL_NAME = str(LOCAL_MODEL_DIR)
print('Model copied to local SSD:', MODEL_NAME)
print({
    'seeds': SEEDS, 'epochs': EPOCHS, 'max_length': MAX_LENGTH,
    'batch_size': BATCH_SIZE, 'gradient_accumulation': GRAD_ACCUM_STEPS,
    'learning_rate': TRAIN_LR, 'reproduction_tolerance_points': REPRO_TOLERANCE_POINTS,
})


## Data preparation

BoolQ's official training split is divided by passage groups. The official validation split is kept untouched as held-out evaluation data. Candidate order is randomized during training and mapped back to semantic yes/no labels during evaluation.


In [ ]:
FAST_MODE = False  # Locked: exact V9 reproduction only.
raw = load_dataset('google/boolq', token=HF_TOKEN)
train_full = raw['train']
heldout_raw = raw['validation']

if FAST_MODE:
    train_full = train_full.shuffle(seed=DATA_SEED).select(range(min(1400, len(train_full))))
    heldout_raw = heldout_raw.shuffle(seed=DATA_SEED).select(range(min(600, len(heldout_raw))))
    val_target = min(240, len(train_full) // 5)
else:
    val_target = VAL_TARGET

groups = np.asarray(train_full['passage'])
labels = np.asarray(train_full['answer'], dtype=int)
test_fraction = val_target / len(train_full)
splitter = GroupShuffleSplit(n_splits=80, test_size=test_fraction, random_state=DATA_SEED)

global_rate = labels.mean()
best = None
for tr_idx, va_idx in splitter.split(np.zeros(len(labels)), labels, groups):
    score = abs(len(va_idx) - val_target) / max(1, val_target) + abs(labels[va_idx].mean() - global_rate)
    if best is None or score < best[0]:
        best = (score, tr_idx, va_idx)
_, train_idx, val_idx = best
train_raw = train_full.select(train_idx.tolist())
val_raw = train_full.select(val_idx.tolist())

assert set(train_raw['passage']).isdisjoint(set(val_raw['passage']))

data_audit = pd.DataFrame([
    {'split': 'train', 'n': len(train_raw), 'yes_rate': np.mean(train_raw['answer']), 'unique_passages': len(set(train_raw['passage']))},
    {'split': 'internal_validation', 'n': len(val_raw), 'yes_rate': np.mean(val_raw['answer']), 'unique_passages': len(set(val_raw['passage']))},
    {'split': 'official_validation_evaluation', 'n': len(heldout_raw), 'yes_rate': np.mean(heldout_raw['answer']), 'unique_passages': len(set(heldout_raw['passage']))},
])
display(data_audit)
data_audit.to_csv(RESULT_DIR / 'data_audit.csv', index=False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, use_fast=True)

def derangement(n, seed):
    rng = np.random.default_rng(seed)
    base = np.arange(n)
    for _ in range(1000):
        perm = rng.permutation(n)
        if np.all(perm != base):
            return perm
    return np.roll(base, 1)

def replace_passages(ds, passages):
    return Dataset.from_dict({'question': ds['question'], 'passage': list(passages), 'answer': ds['answer']})

def encode_dataset(ds, empty_context=False, desc='tokenize'):
    def enc(batch):
        questions = [f'question: {q} answer:' for q in batch['question']]
        passages = ['' for _ in batch['passage']] if empty_context else batch['passage']
        yes_q = [q + ' yes' for q in questions]
        no_q = [q + ' no' for q in questions]
        yes = tokenizer(yes_q, passages, truncation='only_second', max_length=MAX_LENGTH)
        no = tokenizer(no_q, passages, truncation='only_second', max_length=MAX_LENGTH)
        return {
            'yes_input_ids': yes['input_ids'], 'yes_attention_mask': yes['attention_mask'],
            'no_input_ids': no['input_ids'], 'no_attention_mask': no['attention_mask'],
        }
    return ds.map(enc, batched=True, batch_size=256, remove_columns=['question', 'passage'], desc=desc)

encoded = DatasetDict({
    'train': encode_dataset(train_raw, False, 'context train'),
    'validation': encode_dataset(val_raw, False, 'context validation'),
    'evaluation': encode_dataset(heldout_raw, False, 'context evaluation'),
})
empty_evaluation = encode_dataset(heldout_raw, True, 'empty-context evaluation')

shuffled_evaluations = []
for trial in range(SHUFFLE_TRIALS):
    perm = derangement(len(heldout_raw), DATA_SEED + 100 + trial)
    shuffled = replace_passages(heldout_raw, np.asarray(heldout_raw['passage'], dtype=object)[perm])
    shuffled_evaluations.append(encode_dataset(shuffled, False, f'shuffled evaluation {trial + 1}'))

print('Tokenization complete.')


In [ ]:
class PairDataset(torch.utils.data.Dataset):
    def __init__(self, encoded_ds, training=False, seed=0, force_swap=False):
        self.ds = encoded_ds
        self.training = training
        self.seed = int(seed)
        self.epoch = 0
        self.force_swap = force_swap

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        row = self.ds[int(idx)]
        if self.training:
            swap = random.Random(self.seed * 1000003 + self.epoch * 10007 + int(idx)).random() < 0.5
        else:
            swap = self.force_swap
        semantics = [0, 1] if swap else [1, 0]
        prefix = ['no', 'yes'] if swap else ['yes', 'no']
        candidates = [
            {'input_ids': row[f'{p}_input_ids'], 'attention_mask': row[f'{p}_attention_mask']}
            for p in prefix
        ]
        gold_semantic = int(row['answer'])
        return {
            'candidates': candidates,
            'candidate_semantics': semantics,
            'labels': semantics.index(gold_semantic),
            'semantic_labels': gold_semantic,
        }

def pair_collate(batch):
    flat = [candidate for item in batch for candidate in item['candidates']]
    padded = tokenizer.pad(flat, padding=True, return_tensors='pt')
    bsz = len(batch)
    return {
        'input_ids': padded['input_ids'].view(bsz, 2, -1),
        'attention_mask': padded['attention_mask'].view(bsz, 2, -1),
        'candidate_semantics': torch.tensor([x['candidate_semantics'] for x in batch], dtype=torch.long),
        'labels': torch.tensor([x['labels'] for x in batch], dtype=torch.long),
        'semantic_labels': torch.tensor([x['semantic_labels'] for x in batch], dtype=torch.long),
    }

def make_loader(ds, seed, training=False, force_swap=False, batch_size=None):
    pair_ds = PairDataset(ds, training=training, seed=seed, force_swap=force_swap)
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        pair_ds,
        batch_size=batch_size or (BATCH_SIZE if training else EVAL_BATCH_SIZE),
        shuffle=training,
        collate_fn=pair_collate,
        num_workers=2,
        pin_memory=True,
        generator=generator,
        persistent_workers=False,
    )
    return pair_ds, loader


## Frozen encoder and parameter-matched adapters

The encoder is shared by all three alternatives and always runs under `no_grad`. A batch is encoded once, then the same hidden states are sent to all trainable heads. This removes repeated DeBERTa passes and makes the pilot substantially faster than three independent full fine-tuning runs.


In [ ]:
class GatedMLPAdapter(nn.Module):
    def __init__(self, hidden_size, bottleneck, dropout=0.1):
        super().__init__()
        self.down = nn.Linear(hidden_size, bottleneck, bias=False)
        self.up = nn.Linear(bottleneck, hidden_size, bias=False)
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.alpha_logit = nn.Parameter(torch.tensor(math.log(0.1 / 0.9), dtype=torch.float32))

    @property
    def alpha(self):
        return torch.sigmoid(self.alpha_logit)

    def forward(self, hidden, attention_mask=None, enabled=True):
        if not enabled:
            return hidden
        z = self.up(F.gelu(self.down(hidden)))
        return self.norm(hidden + self.alpha * self.dropout(z))

class LatentRelationAdapter(nn.Module):
    def __init__(self, hidden_size, rank, heads, dropout=0.1):
        super().__init__()
        assert rank % heads == 0
        self.rank = rank
        self.heads = heads
        self.head_dim = rank // heads
        self.q = nn.Linear(hidden_size, rank, bias=False)
        self.k = nn.Linear(hidden_size, rank, bias=False)
        self.v = nn.Linear(hidden_size, rank, bias=False)
        self.out = nn.Linear(rank, hidden_size, bias=False)
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.alpha_logit = nn.Parameter(torch.tensor(math.log(0.1 / 0.9), dtype=torch.float32))

    @property
    def alpha(self):
        return torch.sigmoid(self.alpha_logit)

    def forward(self, hidden, attention_mask, enabled=True):
        if not enabled:
            return hidden
        bsz, seq_len, _ = hidden.shape

        def split(x):
            return x.view(bsz, seq_len, self.heads, self.head_dim).transpose(1, 2)

        q, k, v = split(self.q(hidden)), split(self.k(hidden)), split(self.v(hidden))
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(
            attention_mask[:, None, None, :] == 0,
            torch.finfo(scores.dtype).min,
        )
        attn = torch.softmax(scores, dim=-1)
        z = torch.matmul(attn, v).transpose(1, 2).contiguous().view(bsz, seq_len, self.rank)
        return self.norm(hidden + self.alpha * self.dropout(self.out(z)))

class FrozenFeatureScorer(nn.Module):
    def __init__(self, mode, hidden_size):
        super().__init__()
        self.mode = mode
        if mode == 'head_only':
            self.adapter = None
        elif mode == 'mlp_adapter':
            self.adapter = GatedMLPAdapter(hidden_size, MLP_BOTTLENECK)
        elif mode == 'lcca_adapter':
            self.adapter = LatentRelationAdapter(hidden_size, LATENT_RANK, LATENT_HEADS)
        else:
            raise ValueError(mode)
        self.dropout = nn.Dropout(0.15)
        self.scorer = nn.Linear(hidden_size, 1)

    def forward(self, hidden, attention_mask, adapter_enabled=True):
        if self.adapter is not None:
            hidden = self.adapter(hidden, attention_mask, enabled=adapter_enabled)
        return self.scorer(self.dropout(hidden[:, 0]))

encoder = AutoModel.from_pretrained(MODEL_NAME, token=HF_TOKEN).to(DEVICE)
if hasattr(encoder.config, 'use_cache'):
    encoder.config.use_cache = False
for parameter in encoder.parameters():
    parameter.requires_grad = False
encoder.eval()

hidden_size = getattr(encoder.config, 'hidden_size', None)
if hidden_size is None:
    hidden_size = getattr(encoder.config, 'dim')

MODES = ['head_only', 'mlp_adapter', 'lcca_adapter']

def build_models(seed):
    seed_everything(seed)
    return {mode: FrozenFeatureScorer(mode, hidden_size).to(DEVICE) for mode in MODES}

def trainable_count(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

audit_models = build_models(SEEDS[0])
parameter_rows = []
for mode, model in audit_models.items():
    adapter_count = trainable_count(model.adapter) if model.adapter is not None else 0
    parameter_rows.append({
        'model': mode,
        'encoder_parameters_frozen': sum(p.numel() for p in encoder.parameters()),
        'adapter_parameters': adapter_count,
        'scorer_parameters': trainable_count(model.scorer),
        'total_trainable_parameters': trainable_count(model),
    })
parameter_df = pd.DataFrame(parameter_rows)
display(parameter_df)
parameter_df.to_csv(RESULT_DIR / 'parameter_counts.csv', index=False)
mlp_n = int(parameter_df.loc[parameter_df.model == 'mlp_adapter', 'adapter_parameters'].iloc[0])
lcca_n = int(parameter_df.loc[parameter_df.model == 'lcca_adapter', 'adapter_parameters'].iloc[0])
assert mlp_n == lcca_n, (mlp_n, lcca_n)
del audit_models
gc.collect()
torch.cuda.empty_cache()
print('MLP and LCCA adapter parameters match exactly:', mlp_n)


In [ ]:
def semantic_metrics(gold, pred, loss=None):
    return {
        'loss': float(loss) if loss is not None else np.nan,
        'accuracy': accuracy_score(gold, pred),
        'balanced_accuracy': balanced_accuracy_score(gold, pred),
        'macro_f1': f1_score(gold, pred, average='macro', zero_division=0),
        'predicted_yes_rate': float(np.mean(pred)),
        'true_yes_rate': float(np.mean(gold)),
    }

def encode_batch(batch):
    ids = batch['input_ids'].to(DEVICE, non_blocking=True)
    mask = batch['attention_mask'].to(DEVICE, non_blocking=True)
    bsz, nopt, seq_len = ids.shape
    flat_ids = ids.view(bsz * nopt, seq_len)
    flat_mask = mask.view(bsz * nopt, seq_len)
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=AMP_DTYPE):
        hidden = encoder(input_ids=flat_ids, attention_mask=flat_mask).last_hidden_state
    return hidden.detach(), flat_mask, bsz, nopt

@torch.no_grad()
def evaluate_variants(variants, loader, return_predictions=False):
    for model, _ in variants.values():
        model.eval()
    totals = {name: {'loss_sum': 0.0, 'gold': [], 'pred': []} for name in variants}
    for batch in loader:
        hidden, flat_mask, bsz, nopt = encode_batch(batch)
        labels = batch['labels'].to(DEVICE)
        semantics = batch['candidate_semantics']
        gold = batch['semantic_labels'].numpy()
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            for name, (model, enabled) in variants.items():
                logits = model(hidden, flat_mask, adapter_enabled=enabled).view(bsz, nopt)
                loss = F.cross_entropy(logits.float(), labels)
                pred_index = logits.argmax(dim=-1).cpu()
                semantic_pred = semantics.gather(1, pred_index[:, None]).squeeze(1).numpy()
                totals[name]['loss_sum'] += loss.item() * bsz
                totals[name]['gold'].extend(gold.tolist())
                totals[name]['pred'].extend(semantic_pred.tolist())

    output = {}
    for name, values in totals.items():
        gold = np.asarray(values['gold'])
        pred = np.asarray(values['pred'])
        metrics = semantic_metrics(gold, pred, values['loss_sum'] / len(gold))
        if return_predictions:
            metrics['gold'] = gold
            metrics['pred'] = pred
        output[name] = metrics
    return output

yes_count = int(np.sum(encoded['train']['answer']))
no_count = len(encoded['train']) - yes_count
class_weight = torch.tensor([
    len(encoded['train']) / (2 * max(no_count, 1)),
    len(encoded['train']) / (2 * max(yes_count, 1)),
], dtype=torch.float32, device=DEVICE)

def train_seed(seed):
    seed_everything(seed)
    models = build_models(seed)
    train_ds, train_loader = make_loader(encoded['train'], seed, training=True)
    _, val_loader = make_loader(encoded['validation'], seed)

    optimizers = {
        mode: torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=TRAIN_LR,
            weight_decay=WEIGHT_DECAY,
        )
        for mode, model in models.items()
    }
    updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
    total_updates = updates_per_epoch * EPOCHS
    schedulers = {
        mode: get_linear_schedule_with_warmup(
            optimizer, int(total_updates * WARMUP_RATIO), total_updates
        )
        for mode, optimizer in optimizers.items()
    }
    scaler = torch.amp.GradScaler('cuda', enabled=(AMP_DTYPE == torch.float16))
    best = {mode: {'key': None, 'state': None, 'epoch': None} for mode in MODES}
    history = []
    start = time.time()
    torch.cuda.reset_peak_memory_stats()

    for epoch in range(1, EPOCHS + 1):
        train_ds.set_epoch(epoch)
        encoder.eval()
        for model in models.values():
            model.train()
        for optimizer in optimizers.values():
            optimizer.zero_grad(set_to_none=True)
        stats = {mode: {'loss': 0.0, 'correct': 0, 'seen': 0} for mode in MODES}

        for step, batch in enumerate(train_loader, 1):
            hidden, flat_mask, bsz, nopt = encode_batch(batch)
            labels = batch['labels'].to(DEVICE)
            semantic = batch['semantic_labels'].to(DEVICE)
            combined_loss = 0.0
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                for mode, model in models.items():
                    logits = model(hidden, flat_mask, adapter_enabled=True).view(bsz, nopt)
                    per_item = F.cross_entropy(logits.float(), labels, reduction='none')
                    loss = (per_item * class_weight[semantic]).mean()
                    combined_loss = combined_loss + loss
                    stats[mode]['loss'] += loss.item() * bsz
                    stats[mode]['correct'] += (logits.argmax(-1) == labels).sum().item()
                    stats[mode]['seen'] += bsz

            scaler.scale(combined_loss / GRAD_ACCUM_STEPS).backward()
            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
                for mode in MODES:
                    scaler.unscale_(optimizers[mode])
                    torch.nn.utils.clip_grad_norm_(models[mode].parameters(), 1.0)
                    scaler.step(optimizers[mode])
                scaler.update()
                for mode in MODES:
                    optimizers[mode].zero_grad(set_to_none=True)
                    schedulers[mode].step()

            if step % 500 == 0:
                print(f'Seed {seed}, epoch {epoch}/{EPOCHS}, batch {step}/{len(train_loader)}')

        val_results = evaluate_variants(
            {mode: (models[mode], True) for mode in MODES}, val_loader
        )
        for mode in MODES:
            val = val_results[mode]
            row = {
                'model': mode,
                'seed': seed,
                'epoch': epoch,
                'train_loss': stats[mode]['loss'] / stats[mode]['seen'],
                'train_accuracy': stats[mode]['correct'] / stats[mode]['seen'],
                **{f'val_{k}': v for k, v in val.items()},
            }
            history.append(row)
            print(mode, seed, {k: round(v, 5) if isinstance(v, float) else v for k, v in row.items()})
            key = (val['balanced_accuracy'], val['macro_f1'], -val['loss'])
            if best[mode]['key'] is None or key > best[mode]['key']:
                best[mode] = {
                    'key': key,
                    'state': {k: v.detach().cpu().clone() for k, v in models[mode].state_dict().items()},
                    'epoch': epoch,
                }

    for mode in MODES:
        models[mode].load_state_dict(best[mode]['state'])
        models[mode].to(DEVICE).eval()
    return {
        'models': models,
        'best': best,
        'history': history,
        'training_seconds': time.time() - start,
        'peak_gpu_gb': torch.cuda.max_memory_allocated() / (1024 ** 3),
    }


## Held-out evaluation and interventions

Normal, shuffled, empty-context, and option-order evaluations reuse the same frozen encoder batch for all alternatives. Adapter-off variants bypass only the trained residual module without retraining.


In [ ]:
import hashlib

MODE_ALIASES = {'head_only': 'head', 'mlp_adapter': 'mlp', 'lcca_adapter': 'lcca'}

def checkpoint_name(mode, seed):
    return f'{MODE_ALIASES[mode]}_seed{seed}.pt'

def cpu_state_dict(module):
    if module is None:
        return None
    return {k: v.detach().cpu().clone() for k, v in module.state_dict().items()}

def save_checkpoint(model, mode, seed, best_epoch):
    payload = {
        'format_version': 1,
        'experiment': 'lcca_v9_exact_reproduction',
        'mode': mode,
        'alias': MODE_ALIASES[mode],
        'seed': int(seed),
        'best_epoch': int(best_epoch),
        'base_encoder': 'deberta-v3-large',
        'encoder_frozen': True,
        'hidden_size': int(hidden_size),
        'latent_rank': LATENT_RANK,
        'latent_heads': LATENT_HEADS,
        'mlp_bottleneck': MLP_BOTTLENECK,
        'max_length': MAX_LENGTH,
        'input_serialization': 'tokenizer(question_prefix_plus_option, passage)',
        'adapter_state_dict': cpu_state_dict(model.adapter),
        'scorer_state_dict': cpu_state_dict(model.scorer),
    }
    name = checkpoint_name(mode, seed)
    local_path = CHECKPOINT_DIR / name
    drive_path = DRIVE_CHECKPOINT_DIR / name
    torch.save(payload, local_path)
    shutil.copy2(local_path, drive_path)
    return local_path

def load_partial(filename):
    path = PROGRESS_DIR / filename
    if RESUME_FROM_DRIVE and path.is_file():
        return pd.read_csv(path).to_dict('records')
    return []

runtime_rows = load_partial('runtime_partial.csv')
completed_seeds = {int(row['seed']) for row in runtime_rows}

def load_committed(filename):
    return [row for row in load_partial(filename) if int(row['seed']) in completed_seeds]

all_history = load_committed('training_history_partial.csv')
run_rows = load_committed('run_results_partial.csv')
prediction_rows = load_committed('predictions_partial.csv')

if completed_seeds:
    print('Drive checkpoint found. Completed seeds:', sorted(completed_seeds))

for seed in SEEDS:
    if seed in completed_seeds:
        expected = [DRIVE_CHECKPOINT_DIR / checkpoint_name(mode, seed) for mode in MODES]
        missing = [str(p) for p in expected if not p.is_file()]
        if missing:
            raise FileNotFoundError(f'Seed {seed} was marked complete but checkpoints are missing: {missing}')
        for drive_path in expected:
            shutil.copy2(drive_path, CHECKPOINT_DIR / drive_path.name)
        print(f'\n===== SEED {seed} already complete; checkpoints restored locally =====')
        continue

    print(f'\n===== SEED {seed} =====')
    trained = train_seed(seed)
    models = trained['models']
    best = trained['best']
    all_history.extend(trained['history'])

    for mode in MODES:
        saved = save_checkpoint(models[mode], mode, seed, best[mode]['epoch'])
        print('Saved checkpoint:', saved.name)

    _, normal_loader = make_loader(encoded['evaluation'], seed)
    normal = evaluate_variants(
        {mode: (models[mode], True) for mode in MODES},
        normal_loader,
        return_predictions=True,
    )

    for mode in MODES:
        run_rows.append({
            'model': mode,
            'seed': seed,
            'best_epoch': best[mode]['epoch'],
            'normal_accuracy': normal[mode]['accuracy'],
            'normal_balanced_accuracy': normal[mode]['balanced_accuracy'],
            'normal_macro_f1': normal[mode]['macro_f1'],
            'predicted_yes_rate': normal[mode]['predicted_yes_rate'],
            'true_yes_rate': normal[mode]['true_yes_rate'],
            'adapter_alpha': (
                float(models[mode].adapter.alpha.detach().cpu())
                if models[mode].adapter is not None else np.nan
            ),
        })
        for idx, (gold, pred) in enumerate(zip(normal[mode]['gold'], normal[mode]['pred'])):
            prediction_rows.append({
                'model': mode, 'seed': seed, 'example_index': idx,
                'gold': int(gold), 'prediction': int(pred),
            })

    runtime_rows.append({
        'seed': seed,
        'training_minutes': trained['training_seconds'] / 60,
        'peak_allocated_gpu_memory_gb': trained['peak_gpu_gb'],
    })

    # The checkpoint files are copied first. The runtime CSV is the commit log
    # and is written last, so an interrupted seed cannot be marked complete.
    pd.DataFrame(all_history).to_csv(PROGRESS_DIR / 'training_history_partial.csv', index=False)
    pd.DataFrame(run_rows).to_csv(PROGRESS_DIR / 'run_results_partial.csv', index=False)
    pd.DataFrame(prediction_rows).to_csv(PROGRESS_DIR / 'predictions_partial.csv', index=False)
    pd.DataFrame(runtime_rows).to_csv(PROGRESS_DIR / 'runtime_partial.csv', index=False)
    completed_seeds.add(seed)
    print(f'Seed {seed} completed and committed to Drive.')

    del models, trained, normal
    gc.collect()
    torch.cuda.empty_cache()

history_df = pd.DataFrame(all_history)
run_df = pd.DataFrame(run_rows)
runtime_df = pd.DataFrame(runtime_rows)
prediction_df = pd.DataFrame(prediction_rows)
history_df.to_csv(RESULT_DIR / 'training_history.csv', index=False)
run_df.to_csv(RESULT_DIR / 'table_run_results.csv', index=False)
runtime_df.to_csv(RESULT_DIR / 'runtime_by_seed.csv', index=False)
prediction_df.to_csv(RESULT_DIR / 'evaluation_predictions.csv', index=False)

aggregate_rows = []
for model_name, group in run_df.groupby('model', sort=False):
    row = {'model': model_name, 'runs': len(group)}
    for metric in [
        'normal_accuracy', 'normal_balanced_accuracy', 'normal_macro_f1',
        'predicted_yes_rate', 'adapter_alpha',
    ]:
        row[f'{metric}_mean'] = group[metric].mean()
        row[f'{metric}_sd'] = group[metric].std(ddof=1)
    aggregate_rows.append(row)
aggregate_df = pd.DataFrame(aggregate_rows)
aggregate_df.to_csv(RESULT_DIR / 'table_aggregate_results.csv', index=False)

checkpoint_rows = []
for mode in MODES:
    for seed in SEEDS:
        path = CHECKPOINT_DIR / checkpoint_name(mode, seed)
        if not path.is_file():
            raise FileNotFoundError(path)
        checkpoint_rows.append({
            'model': MODE_ALIASES[mode], 'seed': seed, 'filename': path.name,
            'bytes': path.stat().st_size,
            'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
        })
checkpoint_df = pd.DataFrame(checkpoint_rows)
assert len(checkpoint_df) == 9 and checkpoint_df.filename.nunique() == 9
checkpoint_df.to_csv(RESULT_DIR / 'checkpoint_manifest.csv', index=False)
display(run_df)
display(aggregate_df)
display(checkpoint_df)


In [ ]:
# ----------------------- LOCKED REPRODUCTION AUDIT -----------------------
expected = {
    'LCCA accuracy mean': 0.8409785932721713,
    'LCCA accuracy SD': 0.007191422639941807,
    'LCCA balanced accuracy mean': 0.8389279661123661,
    'LCCA macro-F1 mean': 0.8334419419978355,
    'LCCA-minus-MLP accuracy': 0.054332313965341515,
    'LCCA-minus-MLP balanced accuracy': 0.07118048030721336,
    'LCCA-minus-MLP macro-F1': 0.06285990501904089,
}

lcca = run_df[run_df.model == 'lcca_adapter'].set_index('seed').sort_index()
mlp = run_df[run_df.model == 'mlp_adapter'].set_index('seed').sort_index()
observed = {
    'LCCA accuracy mean': float(lcca.normal_accuracy.mean()),
    'LCCA accuracy SD': float(lcca.normal_accuracy.std(ddof=1)),
    'LCCA balanced accuracy mean': float(lcca.normal_balanced_accuracy.mean()),
    'LCCA macro-F1 mean': float(lcca.normal_macro_f1.mean()),
    'LCCA-minus-MLP accuracy': float((lcca.normal_accuracy - mlp.normal_accuracy).mean()),
    'LCCA-minus-MLP balanced accuracy': float((lcca.normal_balanced_accuracy - mlp.normal_balanced_accuracy).mean()),
    'LCCA-minus-MLP macro-F1': float((lcca.normal_macro_f1 - mlp.normal_macro_f1).mean()),
}

tolerance = REPRO_TOLERANCE_POINTS / 100.0
comparison_rows = []
for metric, expected_value in expected.items():
    observed_value = observed[metric]
    difference = observed_value - expected_value
    comparison_rows.append({
        'metric': metric,
        'published_percent': 100 * expected_value,
        'reproduced_percent': 100 * observed_value,
        'difference_points': 100 * difference,
        'tolerance_points': REPRO_TOLERANCE_POINTS,
        'passed': bool(abs(difference) <= tolerance),
    })

seed_positive = int(np.sum(
    (lcca.normal_balanced_accuracy - mlp.normal_balanced_accuracy).to_numpy(float) > 0
))
comparison_rows.append({
    'metric': 'LCCA balanced accuracy higher than MLP in every seed',
    'published_percent': 3,
    'reproduced_percent': seed_positive,
    'difference_points': seed_positive - 3,
    'tolerance_points': 0,
    'passed': bool(seed_positive == 3),
})
reproduction_df = pd.DataFrame(comparison_rows)
REPRODUCTION_PASSED = bool(reproduction_df.passed.all())
reproduction_df.to_csv(RESULT_DIR / 'reproduction_comparison.csv', index=False)

lines = [
    '| Metric | Published | Reproduced | Difference | Tolerance | Pass |',
    '|---|---:|---:|---:|---:|:---:|',
]
for row in comparison_rows:
    lines.append(
        f"| {row['metric']} | {row['published_percent']:.4f} | "
        f"{row['reproduced_percent']:.4f} | {row['difference_points']:+.4f} | "
        f"{row['tolerance_points']:.2f} | {'PASS' if row['passed'] else 'FAIL'} |"
    )
lines.append('')
lines.append(f'**Overall reproduction decision: {"PASS" if REPRODUCTION_PASSED else "FAIL"}.**')
reproduction_markdown = '\n'.join(lines)
(RESULT_DIR / 'reproduction_comparison.md').write_text(reproduction_markdown + '\n', encoding='utf-8')
print(reproduction_markdown)

serialization_note = (
    'Executable V9 serialization retained for exact reproduction:\n'
    'tokenizer(first = "question: {question} answer: {option}", second = passage)\n'
    'DeBERTa special-token order: [CLS] question answer option [SEP] passage [SEP]\n'
    'This differs from the passage-first sequence stated later in manuscript prose.\n'
)
(RESULT_DIR / 'input_serialization_audit.txt').write_text(serialization_note, encoding='utf-8')

if not REPRODUCTION_PASSED:
    failure_zip = shutil.make_archive(
        '/content/lcca_v9_reproduction_FAILED_artifacts', 'zip', root_dir=RESULT_DIR
    )
    from google.colab import files
    files.download(failure_zip)
    raise RuntimeError(
        'V9 reproduction audit FAILED. Contrast scoring was intentionally not run. '
        'Share the downloaded failure ZIP before interpreting the paper.'
    )
print('Reproduction PASS. Proceeding to polarity contrast evaluation.')


In [ ]:
# ----------------------- CONTRAST SET: UNCHANGED METRICS -----------------------
import importlib.util

if not SCORE_SCRIPT_PATH.is_file() or not CONTRAST_SET_PATH.is_file():
    raise FileNotFoundError('score_contrast_set.py and contrast_set.jsonl must exist in /content')

# Import the supplied file as a module. No metric/statistical function is edited.
spec = importlib.util.spec_from_file_location('score_contrast_set_user', SCORE_SCRIPT_PATH)
score_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(score_module)
contrast_rows = score_module.load(CONTRAST_SET_PATH)
if len(contrast_rows) != 81:
    raise ValueError(f'Expected 81 contrast pairs, found {len(contrast_rows)}')

# Archive the exact user-supplied metric source and contrast set.
shutil.copy2(SCORE_SCRIPT_PATH, RESULT_DIR / 'score_contrast_set.py')
shutil.copy2(CONTRAST_SET_PATH, RESULT_DIR / 'contrast_set.jsonl')
metric_source_sha256 = hashlib.sha256(SCORE_SCRIPT_PATH.read_bytes()).hexdigest()

# Precompute the frozen encoder output once for each unique candidate input.
# This retains the executable V9 serialization used during training.
unique_inputs = []
seen_inputs = set()
for row in contrast_rows:
    for arm in score_module.ARMS:
        passage = row[f'passage_{arm}']
        question = row['question']
        for option in score_module.OPTIONS:
            key = (passage, question, option)
            if key not in seen_inputs:
                seen_inputs.add(key)
                unique_inputs.append(key)

feature_cache = {}
encoder.eval()
for start in range(0, len(unique_inputs), EVAL_BATCH_SIZE):
    batch_keys = unique_inputs[start:start + EVAL_BATCH_SIZE]
    first = [f'question: {q} answer: {o}' for _, q, o in batch_keys]
    second = [p for p, _, _ in batch_keys]
    tokens = tokenizer(
        first,
        second,
        truncation='only_second',
        max_length=MAX_LENGTH,
        padding=True,
        return_tensors='pt',
    )
    ids = tokens['input_ids'].to(DEVICE)
    mask = tokens['attention_mask'].to(DEVICE)
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=AMP_DTYPE):
        hidden = encoder(input_ids=ids, attention_mask=mask).last_hidden_state
    for i, key in enumerate(batch_keys):
        length = int(mask[i].sum().item())
        feature_cache[key] = (
            hidden[i:i+1, :length].detach().cpu().clone(),
            mask[i:i+1, :length].detach().cpu().clone(),
        )
    if (start // EVAL_BATCH_SIZE + 1) % 25 == 0:
        print(f'Frozen feature cache: {min(start + EVAL_BATCH_SIZE, len(unique_inputs))}/{len(unique_inputs)}')

print('Unique contrast candidate inputs cached:', len(feature_cache))

def safe_torch_load(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        return torch.load(path, map_location='cpu')

loaded_models = {}
MODELS = {}

def make_score_fn(model):
    @torch.no_grad()
    def score_fn(passage, question, option):
        key = (passage, question, option)
        if key not in feature_cache:
            raise KeyError('Contrast input was not present in the frozen feature cache')
        hidden_cpu, mask_cpu = feature_cache[key]
        hidden = hidden_cpu.to(DEVICE, non_blocking=True)
        mask = mask_cpu.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            scalar = model(hidden, mask, adapter_enabled=True).reshape(-1)[0]
        return float(scalar.float().cpu())
    return score_fn

alias_to_mode = {'head': 'head_only', 'mlp': 'mlp_adapter', 'lcca': 'lcca_adapter'}
for alias, mode in alias_to_mode.items():
    for seed in SEEDS:
        ckpt_path = CHECKPOINT_DIR / f'{alias}_seed{seed}.pt'
        checkpoint = safe_torch_load(ckpt_path)
        if checkpoint['mode'] != mode or int(checkpoint['seed']) != seed:
            raise ValueError(f'Checkpoint metadata mismatch: {ckpt_path}')
        model = FrozenFeatureScorer(mode, hidden_size).to(DEVICE)
        if model.adapter is not None:
            model.adapter.load_state_dict(checkpoint['adapter_state_dict'], strict=True)
        model.scorer.load_state_dict(checkpoint['scorer_state_dict'], strict=True)
        model.eval()
        loaded_models[(alias, seed)] = model
        MODELS[(alias, seed)] = make_score_fn(model)

if len(MODELS) != 9:
    raise RuntimeError(f'Expected 9 registered scoring functions, found {len(MODELS)}')
print('Registered MODELS:', sorted(MODELS))

metrics_path = RESULT_DIR / 'metrics.json'
contrast_result = score_module.run(
    contrast_rows,
    MODELS,
    treatment='lcca',
    control='mlp',
    out_path=str(metrics_path),
)
markdown_table = contrast_result['markdown_table']
(RESULT_DIR / 'contrast_markdown_table.md').write_text(markdown_table + '\n', encoding='utf-8')
(RESULT_DIR / 'contrast_metric_source_sha256.txt').write_text(
    metric_source_sha256 + '  score_contrast_set.py\n', encoding='utf-8'
)
print('\nMarkdown table saved to contrast_markdown_table.md')


In [ ]:
# ZIP checkpoints, reproduction audit, and contrast outputs; download automatically.
required_outputs = [
    RESULT_DIR / 'reproduction_comparison.csv',
    RESULT_DIR / 'reproduction_comparison.md',
    RESULT_DIR / 'metrics.json',
    RESULT_DIR / 'contrast_markdown_table.md',
    RESULT_DIR / 'checkpoint_manifest.csv',
] + [CHECKPOINT_DIR / f'{alias}_seed{seed}.pt' for alias in ('head', 'mlp', 'lcca') for seed in SEEDS]
missing_outputs = [str(p) for p in required_outputs if not p.is_file()]
if missing_outputs:
    raise FileNotFoundError(f'Required artifacts missing: {missing_outputs}')

run_config = {
    'experiment': 'lcca_v9_exact_reproduction_checkpoints_and_contrast',
    'base_encoder': str(DRIVE_MODEL_DIR),
    'encoder_frozen': True,
    'seeds': SEEDS,
    'epochs': EPOCHS,
    'max_length': MAX_LENGTH,
    'batch_size': BATCH_SIZE,
    'gradient_accumulation_steps': GRAD_ACCUM_STEPS,
    'learning_rate': TRAIN_LR,
    'weight_decay': WEIGHT_DECAY,
    'warmup_ratio': WARMUP_RATIO,
    'latent_rank': LATENT_RANK,
    'latent_heads': LATENT_HEADS,
    'mlp_bottleneck': MLP_BOTTLENECK,
    'adapter_parameters_each': mlp_n,
    'passage_grouped_split': True,
    'train_examples': len(train_raw),
    'validation_examples': len(val_raw),
    'heldout_examples': len(heldout_raw),
    'reproduction_tolerance_points': REPRO_TOLERANCE_POINTS,
    'reproduction_passed': REPRODUCTION_PASSED,
    'contrast_pairs': len(contrast_rows),
    'registered_models': len(MODELS),
    'input_serialization': 'tokenizer(question_prefix_plus_option, passage)',
    'metric_source_sha256': metric_source_sha256,
}
(RESULT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')

zip_base = '/content/lcca_v9_reproduction_checkpoints_contrast_artifacts'
zip_path = shutil.make_archive(zip_base, 'zip', root_dir=RESULT_DIR)
print('ZIP created:', zip_path)

from google.colab import files
files.download(zip_path)
print('Download triggered. Runtime was NOT disconnected or deleted.')
